In [ ]:
import io
import os
import imageio
import cv2
import random
import collections
from collections import Counter
import ipywidgets
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [ ]:
# 재현을 위한 시드 설정
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    tf.random.set_seed(seed)

seed_everything()

## Hyperparameters

In [ ]:
loaded_parameters = {}

with open('parameters.txt', 'r') as file:
    lines = file.readlines()
    for line in lines:
        key, value = line.strip().split(": ", 1)  # 한 번만 분할하도록 수정
        if value.startswith("{") and value.endswith("}"):
            value = eval(value)  # 딕셔너리 형태의 값은 eval() 사용
        else:
            try:
                value = eval(value)  # 다른 값들은 eval() 사용 시도
            except:
                value = value  # eval()로 변환할 수 없는 경우 문자열 그대로 사용
        loaded_parameters[key] = value

print(loaded_parameters)

In [ ]:
## 불러와지지 않을 경우 사용
# DATA
DATA_PATH = "action"         # 데이터 경로
NUM_CLASSES = 3                # 데이터셋 클래스 개수
SPLIT_RATIOS = {"train": 0.7, "val": 0.15, "test": 0.15}
frame_interval = 5
min_frame_count = float('12')  # 각 비디오에서 가져올 최소 프레임 수 <- 구축
image_size = (60,60)

BATCH_SIZE = 16                 # BATCH_SIZE: 한 번의 forward/backward pass에서 처리되는 샘플의 수를 결정
AUTO = tf.data.AUTOTUNE         # tf.data.AUTOTUNE): TensorFlow가 자동으로 데이터 프리페칭 버퍼 크기를 조정하여 훈련 중 CPU와 GPU 활용도를 최적화하는 기능
INPUT_SHAPE = (12, 60, 60, 3)   # 3D image size는 28x28x28, 단일채널(grayscale) <- 학습

# OPTIMIZER
LEARNING_RATE = 1e-4            # train optimizer의 학습률 지정 : 0.0001
WEIGHT_DECAY = 1e-5             # 가중치 감쇠 지정(과적합을 방지하기 위한 정규화 기법) : 0.00001

# TRAINING
EPOCHS = 60                     # 반복학습 : 60

# TUBELET EMBEDDING
PATCH_SIZE = (4, 4, 4)                                # patch image size는 8x8x8
NUM_PATCHES = (INPUT_SHAPE[0] // PATCH_SIZE[0]) ** 2  # 패치 수 계산 : {(28x28x28)//(8x8x8) = 3}**2 = 9

# ViViT ARCHITECTURE
LAYER_NORM_EPS = 1e-6           # nomalizaion layer에서 사용되는 epsilson 값 지정 <- 0으로 나누는 것을 피하기 위함
PROJECTION_DIM = 128            # 각 패치는 128차원 벡터로 임베딩
NUM_HEADS = 8                   # multi head-attention에서 각 어텐션 헤드의 개수를 지정 : 8개
NUM_LAYERS = 8                  # transformer 레이어 개수 지정 : 8개

## Data Load

In [ ]:
# 파일명 추출
def list_files_per_class(DATA_PATH):
    files = []
    for root, _, filenames in os.walk(DATA_PATH):
        for filename in filenames:
            files.append(os.path.join(root, filename))
    return files

# 파일명에서 클래스명 추출
def get_class(fname):
    return fname.split('\\')[1]

# 클래스별 파일명 dict 생성
def get_files_per_class(files):
    files_for_class = collections.defaultdict(list)
    for fname in files:
        class_name = get_class(fname)
        files_for_class[class_name].append(fname)
    return files_for_class

# train, val, test 분할
def split_data(data, split_ratio):
    # 결과를 저장할 딕셔너리 초기화
    split_data_by_label = {"train": {}, "test": {}, "valid": {}}

    # 클래스별로 데이터 분할
    for class_label, values in data.items():
        total_samples = len(values)
        train_samples = int(total_samples * split_ratio["train"])
        test_samples = int(total_samples * split_ratio["test"])

        # 데이터 랜덤하게 섞기
        random.shuffle(values)

        # 데이터 분할
        split_data_by_label["train"][class_label] = values[:train_samples]
        split_data_by_label["test"][class_label] = values[train_samples:train_samples + test_samples]
        split_data_by_label["valid"][class_label] = values[train_samples + test_samples:]

    return split_data_by_label

# video에서 image 추출 후 배열 변환
def video_to_array(DATA_PATH, min_frame_count):
    files = list_files_per_class(DATA_PATH)

    for f in files:
        tokens = f.split('\\')
        if len(tokens) <= 2:
                files.remove(f)

    files_for_class = get_files_per_class(files)
    labels = list(files_for_class.keys())[:5]

    array_for_video = collections.defaultdict(list)

    for label in labels:
        video_files = files_for_class[label]

        for video_file in video_files:
            vidcap = cv2.VideoCapture(video_file)

            frame_array = []  # 각 비디오의 프레임 배열

            success, frame = vidcap.read()
            frame_count = 0  # 현재 프레임
            frame_count_save = 0 # 저장 프레임 개수
            
            while success:
                if frame_count % frame_interval == 0 and frame_count_save < min_frame_count:
                    images_tensor = tf.convert_to_tensor(frame, dtype=tf.uint8)
                    resized_image = tf.image.resize(images_tensor, 
                                                    image_size, 
                                                    method=tf.image.ResizeMethod.BILINEAR)
                    image_tensor = tf.stack(resized_image)
                    image_array = image_tensor.numpy()
                    frame_array.append(image_array)
                    frame_count_save += 1
                elif frame_count_save > min_frame_count:
                    break  # 최소 프레임 개수를 초과하면 루프 종료
                frame_count += 1
                success, frame = vidcap.read()

            # 최소 프레임 개수를 설정한 경우에만 추가
            if frame_count_save >= min_frame_count:
                array_for_video[label].append(frame_array)
                if frame_count_save < min_frame_count:
                    min_frame_count = frame_count_save
    return array_for_video

## 실행

In [ ]:
array_for_video = video_to_array(DATA_PATH, min_frame_count)

In [ ]:
data_dict = split_data(array_for_video,SPLIT_RATIOS)

In [ ]:
for key, labels in data_dict.items():
    
    # 이미지를 저장할 디렉토리와 파일명
    output_filename = f'inputdata/{key}_with_labels_second.npz'
    
    video_arrays = []
    label_arrays = []
    
    for label,value in labels.items():
        video_arrays.extend(value)
        label_index = list(labels.keys()).index(label)
        label_arrays.extend([label_index] * len(value))
    
    # 비디오 배열과 라벨 배열을 하나의 .npz 파일로 저장
    np.savez(output_filename, videos=video_arrays, labels=label_arrays, dtype=object)